## Synthetic Data Generator
This Data generator uses a large language model and a Gradio interface to allows users to dynamically generate realistic tabular datasets in CSV format based on a specified topic, column structure, and number of rows. It's designed to run on Colab only.

The application constructs a structured prompt that instructs the model to output strictly formatted CSV data, which is then parsed into a pandas DataFrame for preview and download. The generated dataset can be exported as a CSV file for further analysis or experimentation.

Key Features

- Generate synthetic datasets from natural language input

- Customizable topic, columns, and dataset size

- Automatic CSV validation and parsing

- Data preview within a Gradio UI

- One-click CSV download

- Designed for rapid prototyping and testing ML/data workflows

Use Cases

- Testing data pipelines

- Creating mock datasets

- Prototyping machine learning experiments

- Educational and demonstration purposes

In [1]:
!pip install -q transformers accelerate bitsandbytes==0.46.1 gradio pandas

In [2]:
import torch
import pandas as pd
import gradio as gr
import gc
import os
from datetime import datetime

from google.colab import userdata
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    BitsAndBytesConfig
)

In [3]:
hf_token = userdata.get("HF_TOKEN")
login(hf_token)

print("GPU available:", torch.cuda.is_available())

GPU available: True


In [4]:
MODEL = "microsoft/Phi-3-mini-4k-instruct" #you can change the model to any other model

In [5]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map="auto",
    quantization_config=quant_config
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [6]:
def prompt_builder(topic, columns, rows):

    system_prompt = """
You are a synthetic data generator.
Generate realistic datasets only.
Output ONLY CSV data.
No explanations.
"""

    example = """
Example CSV:

id,name,species,category
1,Lion,Mammal,Savannah
2,Penguin,Bird,Antarctic
3,Elephant,Mammal,Forest
"""

    user = f"""
Create a synthetic dataset.

Topic: {topic}
Columns: {columns}
Rows: {rows}

Follow the example format EXACTLY.
Return ONLY CSV text.
"""

    return system_prompt + example + user

In [9]:
def generate_dataset(topic, columns, rows):
    prompt = prompt_builder(topic, columns, rows)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = output[0][inputs.input_ids.shape[-1]:]

    text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return text.strip()

In [10]:
import re
from io import StringIO
import pandas as pd

def clean_csv_text(csv_text):

    lines = [l.strip() for l in csv_text.splitlines() if l.strip()]


    header_idx = next((i for i,l in enumerate(lines) if "," in l), None)

    if header_idx is None:
        raise ValueError("No CSV header detected")

    header = lines[header_idx]
    num_cols = len(header.split(","))

    valid_lines = [header]

    for line in lines[header_idx+1:]:
        if len(line.split(",")) == num_cols:
            valid_lines.append(line)

    return "\n".join(valid_lines)



In [11]:
def csv_to_dataframe_converter(csv_text):
    csv_text = clean_csv_text(csv_text)
    try:
        df = pd.read_csv(StringIO(csv_text), sep=",")
    except Exception as e:
        raise ValueError(f"Failed to parse CSV: {e}\nPreview:\n{csv_text[:500]}")
    return df

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive", force_remount=True)

In [12]:
def save_dataset(df, filename_prifix="synthetic_data"):
    # folder_path = "/content/drive/MyDrive/llms/datasets"
    folder_path = "/content/sample_data/datasets"
    os.makedirs(folder_path, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{filename_prifix}_{timestamp}.csv"
    path = os.path.join(folder_path, filename)
    df.to_csv(path, index=False)
    print(f"Saved CSV to {path}")
    return path

In [13]:
def app(topic, columns, rows):
    columns_list = [c.strip() for c in columns.split(",") if c.strip()]

    csv_text = generate_dataset(topic, columns_list, rows)

    try:
        df = csv_to_dataframe_converter(csv_text)
    except ValueError as e:
        return str(e), None

    # Save to Drive
    file_path =  save_dataset(df)

    return df,file_path

In [19]:
with gr.Blocks(theme=gr.themes.Ocean() , title="Synthetic Data Generator") as demo:

    gr.Markdown(
        """
        # Synthetic Data Generator

        Generate realistic synthetic datasets instantly using AI.

        **Steps**
        1. Enter a dataset topic
        2. Specify columns (comma separated)
        3. Choose number of rows
        4. Click **Generate Dataset**
        """
    )

    with gr.Row():
        with gr.Column(scale=1):

            topic = gr.Textbox(
                label="Dataset Topic",
                placeholder="e.g. Ecommerce products, Climate data, Customer churn"
            )

            columns = gr.Textbox(
                label="Columns",
                placeholder="id, name, category, price"
            )

            rows = gr.Slider(
                minimum=5,
                maximum=200,
                value=20,
                step=1,
                label="Number of Rows"
            )

            generate_btn = gr.Button(
                " Generate Dataset",
                variant="primary"
            )

        with gr.Column(scale=2):
            preview = gr.Dataframe(
                label="Dataset Preview",
                interactive=False
            )

            download = gr.File(
                label="⬇️ Download CSV"
            )

    generate_btn.click(
        fn=app,
        inputs=[topic, columns, rows],
        outputs=[preview, download]
    )

demo.launch(share=False)

/tmp/ipykernel_3435/164571303.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Ocean() , title="Synthetic Data Generator") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>